In [1]:
import numpy as np
import pandas as pd

# Task 0
Read the dataset from csv file & perform data cleaning - remove all rows, which contains `?` in some columns.
Also check for data correctness (salary & salary $K).

In [4]:
df = pd.read_csv("../data/adult.csv", index_col=0)

df = df.replace("?", np.nan).dropna()

salary_is_correct = (
    (df["salary"].eq(">50K") & df["salary K$"].gt(50))
	| (df["salary"].eq("<=50K") & df["salary K$"].le(50).le(50))
)

df.shape, salary_is_correct.all()

((30162, 12), True)

# Task 1
Print the count of men and women in the dataset.

In [5]:
df["sex"].value_counts()

Male      20380
Female     9782
Name: sex, dtype: int64

# Task 2
Find the average age of men in dataset

In [6]:
df.loc[df["sex"].eq("Male"), "age"].mean()

39.18400392541707

# Task 3
Get the percentage of people from Poland (native-country)

In [7]:
df["native-country"].eq("Poland").mean() * 100

0.18566408063125786

# Task 4
Get the mean and standard deviation of the age for people who earn > 50K per year. After this, get it for those who earn <= 50K.

In [8]:
df.groupby("salary")["age"].agg(["mean", "std"])

,mean,std
salary,,
<=50K,36.60806,13.464631
>50K,43.95911,10.269633


# Task 5
Check, if there are some people without higher education (education: Bachelors, Prof-school, Assoc-acdm, Assoc-voc, Masters, Doctorate), but with > 50K salary

In [9]:
higher_education = df["education"].str.fullmatch(
	"Bachelors|Prof-school|Assoc-acdm|Assoc-voc|Masters|Doctorate"
)

rich_without_higher_education = df.loc[
	(~higher_education) & (df["salary"] == ">50K")
]

rich_without_higher_education.shape[0], not rich_without_higher_education.empty

(3178, True)

# Task 6
Get the statistics of age for each type of education. Use `groupby` and `describe` for this.

In [10]:
df.groupby("education")["age"].describe()

,count,mean,std,min,25%,50%,75%,max
education,,,,,,,,
10th,820.0,37.897561,16.225795,17.0,23.0,36.0,52.0,90.0
11th,1048.0,32.363550,15.089307,17.0,18.0,28.5,43.0,90.0
12th,377.0,32.013263,14.373710,17.0,19.0,28.0,41.0,79.0
1st-4th,151.0,44.622517,14.929051,19.0,33.0,44.0,56.0,81.0
5th-6th,288.0,41.649306,14.754622,17.0,28.0,41.0,53.0,82.0
7th-8th,557.0,47.631957,15.737479,17.0,34.0,49.0,60.0,90.0
9th,455.0,40.303297,15.335754,17.0,28.0,38.0,53.0,90.0
Assoc-acdm,1008.0,37.286706,10.509755,19.0,29.0,36.0,44.0,90.0
Assoc-voc,1307.0,38.246366,11.181253,19.0,30.0,37.0,45.0,84.0


# Task 7
Compare the married and non-married men salaries. Who earns more? (>50K or <=50K)
Married men are those, whom `marital-status` starts with "Married". Others are not.

In [12]:
men = df["sex"].eq("Male")

men_married_status = np.where(
	df.loc[men, "marital-status"].str.startswith("Married"), "married", "not-married"
)

men_salary_counts = pd.crosstab(
	men_married_status, df.loc[men, "salary"]
)

men_salary_percent = pd.crosstab(
	men_married_status,
	df.loc[men, "salary"],
	normalize="index"
) * 100

display(men_salary_counts)
display(men_salary_percent)

salary,<=50K,>50K
row_0,,
married,7052,5723
not-married,6932,673


salary,<=50K,>50K
row_0,,
married,55.201566,44.798434
not-married,91.150559,8.849441


# Task 8
Get the max hours per week some person works. How many people works the same amount of hours per week?

In [14]:
max_hours = df["hours-per-week"].max()
people_with_max_hours = df["hours-per-week"].eq(max_hours).sum()

max_hours, people_with_max_hours

(99, 78)

# Task 9
Analyze the correlation between data in dataset. Understand connected fields in it and print highlight thier connection.

In [23]:
encoded = pd.get_dummies(
	df.drop(columns="salary K$"),
	drop_first=True,
)

salary_corr = (
	encoded
	.corr()["salary_>50K"]
	.drop("salary_>50K")
	.sort_values(key=lambda values: values.abs(), ascending=False)
)

salary_corr.head(15).to_frame("correlation")

,correlation
marital-status_Married-civ-spouse,0.445418
marital-status_Never-married,-0.320053
age,0.241998
hours-per-week,0.229480
relationship_Own-child,-0.226186
sex_Male,0.216699
occupation_Exec-managerial,0.213442
relationship_Not-in-family,-0.193258
occupation_Prof-specialty,0.181458
education_Bachelors,0.178847
